In [16]:
import pandas as pd
import anthropic
from datetime import datetime

# ── 데이터 로드 ─────────────────────────────────────────────
df = pd.read_csv('../weekly_data.csv', encoding='utf-8')

# 이벤트 유무 컬럼 생성 (값 있으면 True)
df['has_event'] = df['event'].notna() & (df['event'].str.strip() != '')

CATEGORIES = ['명품', '패션', '식품', '스포츠', '리빙', '키즈']
TARGETS    = ['명품_target', '패션_target', '식품_target',
              '스포츠_target', '리빙_target', '키즈_target']

sep = "=" * 60

# ── 분석 1: 카테고리별 실적 vs 목표 달성률 ──────────────────
section1_lines = [sep, "1. 카테고리별 실적 vs 목표 달성률", sep]
achievement = {}
for cat, tgt in zip(CATEGORIES, TARGETS):
    actual  = df[cat].sum()
    target  = df[tgt].sum()
    rate    = (actual / target * 100) if target > 0 else 0
    achievement[cat] = {'실적': actual, '목표': target, '달성률': rate}
    section1_lines.append(f"  {cat}: 실적 {actual:>12,.0f}  /  목표 {target:>12,.0f}  =  {rate:.1f}%")

# ── 분석 2: 최근 4주 총매출 추이 ────────────────────────────
section2_lines = [sep, "2. 최근 4주 총매출 추이", sep]
recent = df.tail(4)
for _, row in recent.iterrows():
    section2_lines.append(f"  Week {row['week']} ({row['period_start']}):  {row['total']:>12,.0f}")

# ── 분석 3: 날씨별 카테고리 평균 매출 ───────────────────────
section3_lines = [sep, "3. 날씨별 카테고리 평균 매출", sep]
weather_avg = df.groupby('weather')[CATEGORIES].mean()
header = f"  {'날씨':<6}" + "".join(f"{c:>10}" for c in CATEGORIES)
section3_lines.append(header)
for weather, row in weather_avg.iterrows():
    line = f"  {weather:<6}" + "".join(f"{row[c]:>10,.0f}" for c in CATEGORIES)
    section3_lines.append(line)

# ── 분석 4: 이벤트 유무에 따른 매출 차이 ────────────────────
section4_lines = [sep, "4. 이벤트 유무에 따른 매출 차이", sep]
event_stats = df.groupby('has_event')['total'].agg(['mean', 'count', 'sum'])
for has_ev, row in event_stats.iterrows():
    label = "이벤트 있음" if has_ev else "이벤트 없음"
    section4_lines.append(
        f"  {label}:  평균 {row['mean']:>12,.0f}  |  주차 수 {int(row['count'])}주  |  합계 {row['sum']:>12,.0f}"
    )
if True in event_stats.index and False in event_stats.index:
    diff     = event_stats.loc[True, 'mean'] - event_stats.loc[False, 'mean']
    pct      = diff / event_stats.loc[False, 'mean'] * 100
    section4_lines.append(f"\n  이벤트 효과: +{diff:,.0f} ({pct:+.1f}%)")

# ── 터미널 출력 ──────────────────────────────────────────────
all_sections = (
    section1_lines + [""] +
    section2_lines + [""] +
    section3_lines + [""] +
    section4_lines
)
for line in all_sections:
    print(line)

# ── AI 인사이트 (claude-opus-4-5) ───────────────────────────
print(f"\n{sep}")
print("AI 인사이트 생성 중 (claude-opus-4-5)...")
print(sep)

analysis_text = "\n".join(all_sections)

prompt = f"""다음은 쇼핑몰 주간 판매 데이터 분석 결과입니다.

{analysis_text}

MD팀장 슬랙 보고용으로 아래 내용을 포함해서 **500자 이내**로 작성해주세요:
- 핵심 발견 3가지 (수치 포함)
- 가장 주목할 카테고리와 이유
- 날씨별로 잘 팔리는 카테고리 패턴과 MD 운영 활용 인사이트
- 즉시 액션이 필요한 것 2가지"""

client = anthropic.Anthropic()
response = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=1024,
    messages=[{"role": "user", "content": prompt}]
)
ai_insight = response.content[0].text

print("\n" + ai_insight)

# ── weekly_insight.txt 저장 ──────────────────────────────────
generated_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
output = "\n".join([
    f"주간 자동 보고서  (생성: {generated_at})",
    "",
    *all_sections,
    "",
    sep,
    f"AI 인사이트  (model: claude-opus-4-5)",
    sep,
    ai_insight,
])

with open('weekly_insight.txt', 'w', encoding='utf-8') as f:
    f.write(output)

print(f"\n{sep}")
print(f"✅ 완료! 결과가 'weekly_insight.txt'에 저장되었습니다.")
print(sep)


1. 카테고리별 실적 vs 목표 달성률
  명품: 실적    3,609,000  /  목표    3,588,000  =  100.6%
  패션: 실적    1,729,500  /  목표    1,560,000  =  110.9%
  식품: 실적    1,300,000  /  목표    1,118,000  =  116.3%
  스포츠: 실적      918,100  /  목표      702,000  =  130.8%
  리빙: 실적      544,400  /  목표      520,000  =  104.7%
  키즈: 실적      392,300  /  목표      364,000  =  107.8%

2. 최근 4주 총매출 추이
  Week W23 (2024-06-01):       368,400
  Week W24 (2024-06-08):       377,500
  Week W25 (2024-06-15):       402,300
  Week W26 (2024-06-22):       416,600

3. 날씨별 카테고리 평균 매출
  날씨            명품        패션        식품       스포츠        리빙        키즈
  더움       176,000    79,750    56,125    42,575    21,675    15,075
  맑음       134,471    68,971    49,647    38,147    21,912    16,159
  비        130,500    46,000    46,750    18,650    15,600    10,800
  흐림       119,333    48,667    46,000    20,667    18,000    11,900

4. 이벤트 유무에 따른 매출 차이
  이벤트 없음:  평균      311,622  |  주차 수 18주  |  합계    5,609,200
  이벤트 있음:  평균      360,262  |  주차 수 8주  |

TypeError: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

In [1]:
%pip install anthropic pandas openpyxl python-dotenv

Note: you may need to restart the kernel to use updated packages.
